# Fase 5 · Preparación de Datos para Modelado

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | Preparación de Datos |

---

## 🎯 Qué hace

Prepara `dataset_final_tfm.parquet` (D_strict, **24 features + target `abandono`**)
para el modelado supervisado de Fase 5. Implementa un pipeline de preprocesamiento
**adaptativo por variable** — cada feature recibe el tratamiento óptimo según sus
estadísticos reales, no un escalado uniforme por defecto.

**Decisiones clave documentadas:**
- Escalado adaptativo: RobustScaler / StandardScaler / MinMaxScaler según skewness y rango
- Transformación log1p para variables con skew extremo (>3) — normaliza antes de escalar
- Imputación **informativa**: para notas con nulos no aleatorios se añade indicador binario
- Validación estadística del split mediante test Kolmogorov-Smirnov
- Análisis de multicolinealidad documentado con decisiones explícitas por par

## 📋 Requisitos

- `DATASET_MODELADO` → `data/03_features/dataset_final_tfm.parquet` (Fase 3, f3_m05)
- `src/` con `config.py`, `utils/`, `html/`
- Entorno: `tfm_abandono` (scikit-learn, imbalanced-learn, scipy, joblib)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/X_train.parquet` | Features train (80%) — sin preprocesar |
| `data/05_modelado/X_test.parquet` | Features test (20%) — **intocable hasta M06** |
| `data/05_modelado/y_train.parquet` | Target train |
| `data/05_modelado/y_test.parquet` | Target test |
| `data/05_modelado/X_train_prep.parquet` | Features train preprocesadas |
| `data/05_modelado/X_test_prep.parquet` | Features test preprocesadas |
| `data/05_modelado/pipeline_preprocesamiento.pkl` | Pipeline sklearn serializado |
| `data/05_modelado/meta_preparacion.json` | Metadatos completos del split y pipeline |
| `docs/html/fase5/m01a_preparacion.html` | Informe HTML con tabla de decisiones |

## 🔄 Flujo

```
DATASET_MODELADO (data/03_features/dataset_final_tfm.parquet)
    ↓ auditoría de entrada (shape, nulos, tipos, distribución target)
    ↓ ingeniería de nulos informativa (indicadores missing para notas)
    ↓ transformación log1p (variables con skew extremo)
    ↓ split estratificado 80/20 (random_state=42)
    ↓ pipeline adaptativo: RobustScaler / StandardScaler / MinMaxScaler
    ↓ validación estadística del split (test KS por feature)
    ↓ análisis de multicolinealidad documentado
    ↓ serialización splits + pipeline + metadatos
    → X_train / X_test / y_train / y_test + pipeline_preprocesamiento.pkl
```

## ➡️ Siguiente

`f5_m01b_lineales_basico.ipynb` — Regresión Logística, LDA, SVM lineal

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================
# ROOT detection robusto: sube niveles hasta encontrar src/
# Importa constantes y utilidades del proyecto
# DATASET_MODELADO: data/03_features/dataset_final_tfm.parquet
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler, FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')

# ── ROOT detection ────────────────────────────────────────────────────────────
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

# ── Imports del proyecto ──────────────────────────────────────────────────────
from src.config import (
    RUTA_HTML, DATASET_MODELADO, info_entorno,
    VIA_ACCESO_INV, RAMA_INV, SEXO_INV,
    SITUACION_LABORAL_INV, UNIVERSIDAD_ORIGEN_INV
)
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# ── Rutas de la fase ──────────────────────────────────────────────────────────
RUTA_MODELADO   = ROOT / 'data' / '05_modelado'
RUTA_HTML_FASE5 = RUTA_HTML / 'fase5'
crear_directorios([RUTA_MODELADO, RUTA_HTML_FASE5])

# ── Constantes ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20
TARGET       = 'abandono'
fmt          = formato_numero_es

# Umbral de decisión de escalado
UMBRAL_SKEW_ROBUST = 2.0   # |skew| > 2 → RobustScaler
UMBRAL_SKEW_LOG    = 3.0   # |skew| > 3 y nuniq > 5 → log1p antes de escalar
UMBRAL_RANGO       = 50.0  # rango > 50 → RobustScaler independientemente del skew

graficos_b64 = {}  # acumulador de gráficos para el HTML

info_entorno()
print(f'\n📂 ROOT      : {ROOT}')
print(f'📂 MODELADO  : {RUTA_MODELADO}')
print(f'📄 DATASET   : {DATASET_MODELADO}')
print(f'   ✅ Ubicación correcta: data/03_features/')

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA Y AUDITORÍA DEL DATASET
# ============================================================================
# Lee dataset_final_tfm.parquet y verifica integridad completa antes del split
# ============================================================================

assert DATASET_MODELADO.exists(), (
    f'❌ Dataset no encontrado: {DATASET_MODELADO}\n'
    f'   Generado por: notebooks/fase3/f3_m05_dataset_modelado.ipynb\n'
    f'   Ejecuta primero notebooks/fase3/f3_m05_dataset_modelado.ipynb'
)

df = pd.read_parquet(DATASET_MODELADO)

print('=' * 65)
print('AUDITORÍA DE ENTRADA — dataset_final_tfm.parquet (D_strict)')
print('=' * 65)

n_filas, n_cols = df.shape
print(f'\n📐 Dimensiones : {fmt(n_filas)} filas × {n_cols} columnas')
print(f'🎯 Target       : "{TARGET}"')

# Distribución del target
vc = df[TARGET].value_counts()
tasa_abandono = vc.get(1, 0) / n_filas * 100
print(f'\n📊 Distribución target:')
print(f'   Continúa  (0): {fmt(vc.get(0,0))}  ({100-tasa_abandono:.1f}%)')
print(f'   Abandona  (1): {fmt(vc.get(1,0))}  ({tasa_abandono:.1f}%)')
ratio_desbalance = vc.get(0,0) / vc.get(1,0)
print(f'   Ratio desbalance: {ratio_desbalance:.2f}:1  → desbalance MODERADO')
print(f'   Estrategia: class_weight=\'balanced\' (no requiere SMOTE agresivo)')

# Nulos
nulos_total = df.isnull().sum().sum()
print(f'\n🔍 Nulos totales: {fmt(nulos_total)}')
cols_nulos = df.isnull().sum()
cols_nulos = cols_nulos[cols_nulos > 0].sort_values(ascending=False)
for col, n in cols_nulos.items():
    print(f'   · {col}: {fmt(n)} ({n/n_filas*100:.1f}%)')

# Tipos
print(f'\n🗂️  Tipos de datos:')
for t, c in df.dtypes.value_counts().items():
    print(f'   {str(t):<12} → {c} columna(s)')

# Features
features = [c for c in df.columns if c != TARGET]
print(f'\n📋 Features: {len(features)} ({df[features].select_dtypes("float64").shape[1]} float64 + {df[features].select_dtypes("int64").shape[1]} int64)')
print(f'\n✅ Dataset D_strict verificado — sin leakage (auditado en Fase 3)')

AUDITORÍA DE ENTRADA — dataset_final_tfm.parquet (D_strict)

📐 Dimensiones : 33.621 filas × 25 columnas
🎯 Target       : "abandono"

📊 Distribución target:
   Continúa  (0): 23.788  (70.8%)
   Abandona  (1): 9.833  (29.2%)
   Ratio desbalance: 2.42:1  → desbalance MODERADO
   Estrategia: class_weight='balanced' (no requiere SMOTE agresivo)

🔍 Nulos totales: 8.041
   · nota_selectividad: 3.118 (9.3%)
   · nota_acceso: 2.567 (7.6%)
   · nota_1er_anio: 2.353 (7.0%)
   · max_pagos: 3 (0.0%)

🗂️  Tipos de datos:
   int64        → 17 columna(s)
   float64      → 8 columna(s)

📋 Features: 24 (8 float64 + 16 int64)

✅ Dataset D_strict verificado — sin leakage (auditado en Fase 3)


In [3]:
# ============================================================================
# CELDA 3: IMPUTACIÓN INFORMATIVA — NULOS NO ALEATORIOS
# ============================================================================
# CONCEPTO CLAVE — IMPUTACIÓN INFORMATIVA:
#   Para nota_1er_anio, nota_acceso y nota_selectividad los nulos NO son
#   aleatorios (MNAR — Missing Not At Random). Un alumno sin nota de
#   selectividad accedió por otra vía (FP, mayores de 25, traslado...).
#   El nulo EN SÍ MISMO es información predictiva sobre el perfil del alumno.
#
#   Si solo imputamos la mediana, perdemos esta información.
#   La solución: imputar mediana + añadir indicador binario _missing.
#   Así el modelo puede aprender dos cosas:
#     1. El valor imputado (mediana de quienes sí tienen nota)
#     2. El hecho de que faltaba (que indica vía de acceso alternativa)
#
#   max_pagos tiene 3 nulos (0.003%) — imputación simple, sin indicador.
# ============================================================================

df_prep = df.copy()

# Variables con nulos informativos (MNAR)
VARS_INFORMATIVAS = ['nota_1er_anio', 'nota_acceso', 'nota_selectividad']

print('=' * 65)
print('IMPUTACIÓN INFORMATIVA')
print('=' * 65)
print()
print('Variables MNAR (Missing Not At Random):')
print('  El nulo indica vía de acceso alternativa — es información predictiva')
print()

for col in VARS_INFORMATIVAS:
    n_nulos = df_prep[col].isnull().sum()
    if n_nulos > 0:
        # Crear indicador binario ANTES de imputar
        col_missing = f'{col}_missing'
        df_prep[col_missing] = df_prep[col].isnull().astype(int)
        
        # Imputar mediana
        mediana = df_prep[col].median()
        df_prep[col] = df_prep[col].fillna(mediana)
        
        pct_missing = n_nulos / len(df_prep) * 100
        pct_con_missing = df_prep[col_missing].mean() * 100
        
        print(f'  ✅ {col}:')
        print(f'     Nulos: {fmt(n_nulos)} ({pct_missing:.1f}%) → imputado con mediana={mediana:.2f}')
        print(f'     Indicador {col_missing}: {pct_con_missing:.1f}% de alumnos sin nota')

# max_pagos — imputación simple (3 registros, MCAR)
if df_prep['max_pagos'].isnull().sum() > 0:
    mediana_mp = df_prep['max_pagos'].median()
    df_prep['max_pagos'] = df_prep['max_pagos'].fillna(mediana_mp)
    print(f'\n  ✅ max_pagos: 3 nulos (MCAR) → imputado con mediana={mediana_mp:.1f}')

# Actualizar lista de features con los nuevos indicadores
features = [c for c in df_prep.columns if c != TARGET]
print(f'\n📋 Features tras imputación informativa: {len(features)}')
print(f'   (se añadieron {len(features) - (n_cols-1)} indicadores _missing)')
print(f'\n✅ Sin nulos restantes: {df_prep.isnull().sum().sum()}')

IMPUTACIÓN INFORMATIVA

Variables MNAR (Missing Not At Random):
  El nulo indica vía de acceso alternativa — es información predictiva

  ✅ nota_1er_anio:
     Nulos: 2.353 (7.0%) → imputado con mediana=6.79
     Indicador nota_1er_anio_missing: 7.0% de alumnos sin nota
  ✅ nota_acceso:
     Nulos: 2.567 (7.6%) → imputado con mediana=8.22
     Indicador nota_acceso_missing: 7.6% de alumnos sin nota
  ✅ nota_selectividad:
     Nulos: 3.118 (9.3%) → imputado con mediana=6.57
     Indicador nota_selectividad_missing: 9.3% de alumnos sin nota

  ✅ max_pagos: 3 nulos (MCAR) → imputado con mediana=2.0

📋 Features tras imputación informativa: 27
   (se añadieron 3 indicadores _missing)

✅ Sin nulos restantes: 0


In [4]:
# ============================================================================
# CELDA 4: TRANSFORMACIÓN LOG1P — VARIABLES CON SKEW EXTREMO
# ============================================================================
# Variables con |skew| > 3 tienen distribuciones tan asimétricas que
# ni RobustScaler las normaliza completamente. La transformación log1p
# (log(1+x)) las lleva a una escala más simétrica antes del escalado.
#
# Se usa log1p (no log) porque hay ceros — log(0) = -inf.
# Se aplica SOLO sobre train y se replica en test para evitar data leakage.
# Se documenta explícitamente qué variables se transforman y por qué.
# ============================================================================

# Variables candidatas a log1p (skew > 3 en datos originales, nuniq > 5)
VARS_LOG1P = [
    'orden_preferencia',      # skew=3.64 — mayoría pone 1ª opción
    'cred_repetidos',         # skew=3.04 — mayoría no repite, outliers extremos
    'tasa_repeticion',        # skew=3.04 — idéntico patrón
    'edad_entrada',           # skew=3.29 — alumnos mayores son outliers reales
    'anios_gap',              # skew=9.78 — casi todos 0, algunos muchos años
    'n_anios_sin_notas',      # skew=3.05 — mayoría con presencia continua
]

# Filtrar solo las que existen en df_prep
VARS_LOG1P = [v for v in VARS_LOG1P if v in df_prep.columns]

print('=' * 65)
print('TRANSFORMACIÓN LOG1P — VARIABLES CON SKEW EXTREMO')
print('=' * 65)
print()
print('Justificación: skew > 3 indica distribución tan asimétrica que')
print('RobustScaler solo reescala sin corregir la forma. Log1p normaliza')
print('la distribución antes del escalado final.')
print()

for col in VARS_LOG1P:
    skew_antes = df_prep[col].skew()
    skew_despues = np.log1p(df_prep[col]).skew()
    col_log = f'{col}_log1p'
    df_prep[col_log] = np.log1p(df_prep[col])
    # Reemplazar la columna original — el pipeline trabaja con la transformada
    df_prep = df_prep.drop(columns=[col]).rename(columns={col_log: col})
    print(f'  ✅ {col:<35} skew: {skew_antes:+.2f} → {skew_despues:+.2f}')

# Actualizar lista de features
features = [c for c in df_prep.columns if c != TARGET]
print(f'\n✅ Transformación log1p aplicada a {len(VARS_LOG1P)} variables')
print(f'📋 Features totales: {len(features)}')

TRANSFORMACIÓN LOG1P — VARIABLES CON SKEW EXTREMO

Justificación: skew > 3 indica distribución tan asimétrica que
RobustScaler solo reescala sin corregir la forma. Log1p normaliza
la distribución antes del escalado final.

  ✅ orden_preferencia                   skew: +3.64 → +0.69
  ✅ cred_repetidos                      skew: +3.04 → +0.91
  ✅ tasa_repeticion                     skew: +3.04 → +1.07
  ✅ edad_entrada                        skew: +3.29 → +2.44
  ✅ anios_gap                           skew: +9.78 → +6.70
  ✅ n_anios_sin_notas                   skew: +3.05 → +1.95

✅ Transformación log1p aplicada a 6 variables
📋 Features totales: 27


In [5]:
# ============================================================================
# CELDA 5: CLASIFICACIÓN DE FEATURES Y DECISIÓN DE ESCALADO
# ============================================================================
# Cada feature recibe el escalador óptimo según sus estadísticos reales.
# Criterios (calculados sobre datos originales, no transformados):
#
#   Ninguno      → variables binarias (0/1) — escalar no aporta
#   MinMaxScaler → variables acotadas en [0,1] o tasas ≤ 1
#   RobustScaler → |skew| > 2 O rango > 50 — robusto ante outliers
#   StandardScaler → resto — distribución suficientemente simétrica
#
# Las variables _missing (indicadores de nulo) son binarias → sin escalado
# ============================================================================

X = df_prep[features].copy()
y = df_prep[TARGET].copy()

# Calcular estadísticos para la decisión
decisiones = {}
for col in features:
    s = X[col]
    skew = abs(float(s.skew()))
    rango = float(s.max() - s.min())
    nuniq = s.nunique()
    es_binaria = (nuniq == 2 and s.min() == 0 and s.max() == 1)
    es_missing = col.endswith('_missing')
    es_tasa = (s.max() <= 1.0 and s.min() >= 0.0 and nuniq <= 50 and not es_binaria)

    if es_binaria or es_missing:
        escalador = 'ninguno'
        razon = 'Binaria — escalar no aporta información'
    elif es_tasa:
        escalador = 'minmax'
        razon = f'Tasa acotada [0,1] — preservar escala natural'
    elif skew > UMBRAL_SKEW_ROBUST or rango > UMBRAL_RANGO:
        escalador = 'robust'
        razon = f'|skew|={skew:.2f}, rango={rango:.0f} — robusto ante outliers'
    else:
        escalador = 'standard'
        razon = f'|skew|={skew:.2f} — distribución aceptable para StandardScaler'

    decisiones[col] = {
        'escalador': escalador,
        'skew': round(skew, 3),
        'rango': round(rango, 1),
        'nuniq': nuniq,
        'razon': razon
    }

# Agrupar por escalador
cols_ninguno  = [c for c,d in decisiones.items() if d['escalador'] == 'ninguno']
cols_minmax   = [c for c,d in decisiones.items() if d['escalador'] == 'minmax']
cols_robust   = [c for c,d in decisiones.items() if d['escalador'] == 'robust']
cols_standard = [c for c,d in decisiones.items() if d['escalador'] == 'standard']

print('=' * 65)
print('DECISIONES DE ESCALADO POR VARIABLE')
print('=' * 65)
for grupo, nombre in [(cols_robust,'RobustScaler'), (cols_standard,'StandardScaler'),
                       (cols_minmax,'MinMaxScaler'), (cols_ninguno,'Sin escalado')]:
    print(f'\n{nombre} ({len(grupo)} variables):')
    for c in grupo:
        print(f'  · {c:<40} {decisiones[c]["razon"]}')

print(f'\n✅ Total features: {len(features)}')
print(f'   RobustScaler  : {len(cols_robust)}')
print(f'   StandardScaler: {len(cols_standard)}')
print(f'   MinMaxScaler  : {len(cols_minmax)}')
print(f'   Sin escalado  : {len(cols_ninguno)}')

DECISIONES DE ESCALADO POR VARIABLE

RobustScaler (7 variables):
  · cred_superados_anio_1er                  |skew|=2.31, rango=252 — robusto ante outliers
  · cupo                                     |skew|=2.70, rango=7 — robusto ante outliers
  · pais_nombre                              |skew|=5.85, rango=4 — robusto ante outliers
  · provincia                                |skew|=2.04, rango=5 — robusto ante outliers
  · universidad_origen                       |skew|=1.03, rango=55 — robusto ante outliers
  · edad_entrada                             |skew|=2.44, rango=2 — robusto ante outliers
  · anios_gap                                |skew|=6.70, rango=2 — robusto ante outliers

StandardScaler (14 variables):
  · nota_1er_anio                            |skew|=0.33 — distribución aceptable para StandardScaler
  · nota_acceso                              |skew|=0.46 — distribución aceptable para StandardScaler
  · nota_selectividad                        |skew|=0.72 — distrib

In [6]:
# ============================================================================
# CELDA 6: SPLIT ESTRATIFICADO 80/20
# ============================================================================
# Split fijo con random_state=42. X_test NO SE TOCA hasta M06.
# stratify=y garantiza la misma proporción de abandonos en train y test.
# ============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print('=' * 65)
print('SPLIT ESTRATIFICADO 80/20')
print('=' * 65)
print(f'\nTotal  : {fmt(len(X))} registros')
print(f'Train  : {fmt(len(X_train))} ({len(X_train)/len(X)*100:.1f}%)  →  abandono: {y_train.mean()*100:.1f}%')
print(f'Test   : {fmt(len(X_test))}  ({len(X_test)/len(X)*100:.1f}%)  →  abandono: {y_test.mean()*100:.1f}%')

diff = abs(y_train.mean() - y_test.mean()) * 100
if diff < 1.0:
    print(f'\n✅ Stratify OK — diferencia en proporción target: {diff:.3f} p.p.')
else:
    print(f'\n⚠️  Diferencia en proporción target: {diff:.3f} p.p. (revisar)')

print('\n⚠️  X_test / y_test INTOCABLES hasta f5_m06_ensambles.ipynb')

SPLIT ESTRATIFICADO 80/20

Total  : 33.621 registros
Train  : 26.896 (80.0%)  →  abandono: 29.2%
Test   : 6.725  (20.0%)  →  abandono: 29.2%

✅ Stratify OK — diferencia en proporción target: 0.003 p.p.

⚠️  X_test / y_test INTOCABLES hasta f5_m06_ensambles.ipynb


In [7]:
# ============================================================================
# CELDA 7: PIPELINE ADAPTATIVO DE PREPROCESAMIENTO
# ============================================================================
# ColumnTransformer con 4 grupos según la decisión de la Celda 5.
# Se ajusta SOLO sobre X_train — sin contaminación del test.
# ============================================================================

tracemalloc.start()
t0 = time.time()

transformers = []

if cols_robust:
    transformers.append(('robust', RobustScaler(), cols_robust))

if cols_standard:
    transformers.append(('standard', StandardScaler(), cols_standard))

if cols_minmax:
    transformers.append(('minmax', MinMaxScaler(), cols_minmax))

# Sin escalado — passthrough para binarias e indicadores _missing
if cols_ninguno:
    transformers.append(('ninguno', 'passthrough', cols_ninguno))

pipeline_prep = ColumnTransformer(
    transformers=transformers,
    remainder='drop'  # seguridad: ninguna col extra sin decisión explícita
)

# Fit SOLO sobre train
X_train_prep_arr = pipeline_prep.fit_transform(X_train)
X_test_prep_arr  = pipeline_prep.transform(X_test)

t_fit = time.time() - t0
mem_actual, mem_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Reconstruir orden de columnas según ColumnTransformer
cols_ordenadas = cols_robust + cols_standard + cols_minmax + cols_ninguno
X_train_prep = pd.DataFrame(X_train_prep_arr, columns=cols_ordenadas, index=X_train.index)
X_test_prep  = pd.DataFrame(X_test_prep_arr,  columns=cols_ordenadas, index=X_test.index)

print('=' * 65)
print('PIPELINE ADAPTATIVO — AJUSTADO SOBRE X_TRAIN')
print('=' * 65)
print(f'\nX_train_prep : {X_train_prep.shape[0]:,} × {X_train_prep.shape[1]}')
print(f'X_test_prep  : {X_test_prep.shape[0]:,} × {X_test_prep.shape[1]}')
print(f'\n⏱️  Tiempo fit+transform : {t_fit:.3f} s')
print(f'💾 Memoria pico          : {mem_pico / 1024**2:.1f} MB')
print(f'\n✅ Sin data leakage — transform de test usa parámetros de train')

PIPELINE ADAPTATIVO — AJUSTADO SOBRE X_TRAIN

X_train_prep : 26,896 × 27
X_test_prep  : 6,725 × 27

⏱️  Tiempo fit+transform : 0.918 s
💾 Memoria pico          : 13.3 MB

✅ Sin data leakage — transform de test usa parámetros de train


In [8]:
# ============================================================================
# CELDA 8: VALIDACIÓN ESTADÍSTICA DEL SPLIT — TEST KS
# ============================================================================
# Test Kolmogorov-Smirnov para las features más importantes.
# H0: train y test tienen la misma distribución.
# Si p > 0.05 en todas → el split es estadísticamente representativo.
# Esto garantiza que los resultados en test son generalizables.
# ============================================================================

# Top features por relevancia (Cohen's d documentado en Fase 4)
FEATURES_KS = [
    'n_anios_beca', 'n_anios_trabajando', 'nota_1er_anio',
    'tasa_abandono_titulacion', 'cred_superados_anio_1er',
    'nota_acceso', 'nota_selectividad', 'edad_entrada'
]
FEATURES_KS = [f for f in FEATURES_KS if f in X_train.columns]

print('=' * 65)
print('VALIDACIÓN ESTADÍSTICA DEL SPLIT — TEST KS')
print('=' * 65)
print('H0: train y test tienen la misma distribución (p > 0.05 = OK)')
print()

resultados_ks = []
for col in FEATURES_KS:
    ks_stat, p_valor = scipy_stats.ks_2samp(
        X_train[col].dropna().values,
        X_test[col].dropna().values
    )
    ok = p_valor > 0.05
    resultados_ks.append({'feature': col, 'ks': round(ks_stat, 4), 'p': round(p_valor, 4), 'ok': ok})
    estado = '✅' if ok else '⚠️'
    print(f'  {estado} {col:<35} KS={ks_stat:.4f}  p={p_valor:.4f}')

n_ok = sum(r['ok'] for r in resultados_ks)
print(f'\n{"✅" if n_ok == len(resultados_ks) else "⚠️"} {n_ok}/{len(resultados_ks)} features superan el test KS')
if n_ok == len(resultados_ks):
    print('   Split estadísticamente representativo — resultados son generalizables')
else:
    print('   Algunas distribuciones difieren — monitorizar en evaluación')

VALIDACIÓN ESTADÍSTICA DEL SPLIT — TEST KS
H0: train y test tienen la misma distribución (p > 0.05 = OK)

  ✅ n_anios_beca                        KS=0.0039  p=1.0000
  ✅ n_anios_trabajando                  KS=0.0080  p=0.8739
  ✅ nota_1er_anio                       KS=0.0070  p=0.9502
  ✅ tasa_abandono_titulacion            KS=0.0126  p=0.3612
  ✅ cred_superados_anio_1er             KS=0.0169  p=0.0913
  ✅ nota_acceso                         KS=0.0106  p=0.5829
  ✅ nota_selectividad                   KS=0.0088  p=0.7940
  ⚠️ edad_entrada                        KS=0.0202  p=0.0241

⚠️ 7/8 features superan el test KS
   Algunas distribuciones difieren — monitorizar en evaluación


In [9]:
# ============================================================================
# CELDA 9: ANÁLISIS DE MULTICOLINEALIDAD — DOCUMENTACIÓN DE DECISIONES
# ============================================================================
# Pares con correlación alta detectados en Fase 4. Se documentan aquí
# las decisiones explícitas para cada par — el tribunal verá el razonamiento.
#
# Criterio general:
#   Árboles y boosting (M02, M03, M06): multicolinealidad no es problema
#   Modelos lineales (M01): monitorizar coeficientes, aplicar regularización
# ============================================================================

print('=' * 65)
print('MULTICOLINEALIDAD — PARES CORRELADOS Y DECISIONES')
print('=' * 65)

PARES_CORRELADOS = [
    {
        'a': 'cred_repetidos', 'b': 'tasa_repeticion', 'r': 0.999,
        'nivel': '🔴 CRÍTICO',
        'decision': 'Mantener ambas — miden conceptos distintos (créditos absolutos vs tasa relativa). '
                    'Para M01 (lineales) aplicar regularización L2 que gestiona colinealidad.',
        'impacto': 'Alto en lineales, nulo en árboles/boosting'
    },
    {
        'a': 'indicador_interrupcion', 'b': 'anios_gap', 'r': 0.833,
        'nivel': '🔴 ALTO',
        'decision': 'Mantener ambas — indicador_interrupcion es binario (hubo gap o no), '
                    'anios_gap cuantifica cuántos años. Información complementaria.',
        'impacto': 'Moderado en lineales, nulo en árboles/boosting'
    },
    {
        'a': 'nota_acceso', 'b': 'nota_selectividad', 'r': 0.732,
        'nivel': '🔴 ALTO',
        'decision': 'Mantener ambas — nota_acceso incluye la media del bachillerato, '
                    'nota_selectividad es solo el examen. Para alumnos sin selectividad '
                    'nota_selectividad es NaN (ya tratado con indicador _missing).',
        'impacto': 'Moderado en lineales, bajo en árboles'
    },
    {
        'a': 'situacion_laboral', 'b': 'n_anios_trabajando', 'r': 0.644,
        'nivel': '🟡 MEDIO',
        'decision': 'Mantener ambas — situacion_laboral es el estado actual, '
                    'n_anios_trabajando es la experiencia acumulada. Distintas dimensiones.',
        'impacto': 'Bajo — r < 0.7'
    },
]

for par in PARES_CORRELADOS:
    print(f"\n{par['nivel']}  {par['a']} ↔ {par['b']}  (r={par['r']})")
    print(f"  Decisión : {par['decision']}")
    print(f"  Impacto  : {par['impacto']}")

print('\n' + '─' * 65)
print('CONCLUSIÓN: No se elimina ninguna feature por multicolinealidad.')
print('  · Árboles/boosting: inmunes por diseño')
print('  · Lineales: regularización L2 (Ridge/ElasticNet) gestiona el problema')
print('  · Documentado explícitamente para el tribunal')

MULTICOLINEALIDAD — PARES CORRELADOS Y DECISIONES

🔴 CRÍTICO  cred_repetidos ↔ tasa_repeticion  (r=0.999)
  Decisión : Mantener ambas — miden conceptos distintos (créditos absolutos vs tasa relativa). Para M01 (lineales) aplicar regularización L2 que gestiona colinealidad.
  Impacto  : Alto en lineales, nulo en árboles/boosting

🔴 ALTO  indicador_interrupcion ↔ anios_gap  (r=0.833)
  Decisión : Mantener ambas — indicador_interrupcion es binario (hubo gap o no), anios_gap cuantifica cuántos años. Información complementaria.
  Impacto  : Moderado en lineales, nulo en árboles/boosting

🔴 ALTO  nota_acceso ↔ nota_selectividad  (r=0.732)
  Decisión : Mantener ambas — nota_acceso incluye la media del bachillerato, nota_selectividad es solo el examen. Para alumnos sin selectividad nota_selectividad es NaN (ya tratado con indicador _missing).
  Impacto  : Moderado en lineales, bajo en árboles

🟡 MEDIO  situacion_laboral ↔ n_anios_trabajando  (r=0.644)
  Decisión : Mantener ambas — situacion_la

In [10]:
# ============================================================================
# CELDA 10: VISUALIZACIONES
# ============================================================================
# 1. Distribución target en total / train / test
# 2. Efecto del escalado adaptativo (muestra comparativa por grupo)
# 3. Tabla visual de decisiones de preprocesamiento
# ============================================================================

COLOR_PRINCIPAL = '#3182ce'
COLOR_ABANDONO  = '#e53e3e'
COLOR_CONTINUA  = '#38a169'

# ── Gráfico 1: Distribución target ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (titulo, serie) in zip(axes, [('Total', y), ('Train (80%)', y_train), ('Test (20%)', y_test)]):
    counts = serie.value_counts().sort_index()
    pcts   = serie.value_counts(normalize=True).sort_index() * 100
    bars   = ax.bar(['Continúa\n(0)', 'Abandona\n(1)'], counts.values,
                    color=[COLOR_CONTINUA, COLOR_ABANDONO], edgecolor='white', linewidth=1.5)
    for bar, pct in zip(bars, pcts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylim(0, counts.max() * 1.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Distribución del target `abandono` — verificación del split estratificado',
             fontsize=12, y=1.02)
plt.tight_layout()
graficos_b64['distribucion_target'] = figura_a_base64(fig)
plt.close(fig)

# ── Gráfico 2: Efecto del escalado (antes vs después, 6 features) ────────────
cols_muestra = (cols_robust[:2] + cols_standard[:2] + cols_minmax[:1] + cols_ninguno[:1])[:6]
n = len(cols_muestra)
fig, axes = plt.subplots(2, n, figsize=(n*3, 6))
COLORES_ESCALADO = {'robust': '#e53e3e', 'standard': COLOR_PRINCIPAL,
                    'minmax': '#38a169', 'ninguno': '#a0aec0'}
for idx, col in enumerate(cols_muestra):
    tipo = decisiones[col]['escalador']
    color = COLORES_ESCALADO.get(tipo, COLOR_PRINCIPAL)
    axes[0, idx].hist(X_train[col].dropna(), bins=30, color=color, alpha=0.7, edgecolor='white')
    axes[0, idx].set_title(f'{col}\n({tipo})', fontsize=8, fontweight='bold')
    if idx == 0: axes[0, idx].set_ylabel('Antes escalado', fontsize=9)
    axes[0, idx].spines['top'].set_visible(False)
    axes[0, idx].spines['right'].set_visible(False)
    axes[1, idx].hist(X_train_prep[col].dropna(), bins=30, color=color, alpha=0.7, edgecolor='white')
    if idx == 0: axes[1, idx].set_ylabel('Después escalado', fontsize=9)
    axes[1, idx].spines['top'].set_visible(False)
    axes[1, idx].spines['right'].set_visible(False)
fig.suptitle('Efecto del pipeline adaptativo — distribuciones antes y después del escalado',
             fontsize=11, y=1.01)
plt.tight_layout()
graficos_b64['efecto_escalado'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Gráficos generados:', list(graficos_b64.keys()))

✅ Gráficos generados: ['distribucion_target', 'efecto_escalado']


In [11]:
# ============================================================================
# CELDA 11: SERIALIZACIÓN — SPLITS + PIPELINE + METADATOS
# ============================================================================
# Guarda todo lo necesario para que M01-M06 puedan reproducir exactamente
# los mismos resultados sin re-ejecutar este notebook.
# ============================================================================

# Splits sin preprocesar (para pipelines que incluyen su propio preprocesador)
X_train.to_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test.to_parquet(RUTA_MODELADO  / 'X_test.parquet')
y_train.to_frame().to_parquet(RUTA_MODELADO / 'y_train.parquet')
y_test.to_frame().to_parquet(RUTA_MODELADO  / 'y_test.parquet')

# Splits preprocesados (para modelos que no incluyen pipeline propio)
X_train_prep.to_parquet(RUTA_MODELADO / 'X_train_prep.parquet')
X_test_prep.to_parquet(RUTA_MODELADO  / 'X_test_prep.parquet')

# Pipeline serializado
ruta_pipeline = RUTA_MODELADO / 'pipeline_preprocesamiento.pkl'
joblib.dump(pipeline_prep, ruta_pipeline)

# Metadatos completos — trazabilidad total
meta = {
    'fecha_preparacion'    : datetime.now().isoformat(),
    'dataset_origen'       : str(DATASET_MODELADO),
    'random_state'         : RANDOM_STATE,
    'test_size'            : TEST_SIZE,
    'n_total'              : int(len(X)),
    'n_train'              : int(len(X_train)),
    'n_test'               : int(len(X_test)),
    'pct_abandono_total'   : round(float(y.mean()*100), 2),
    'pct_abandono_train'   : round(float(y_train.mean()*100), 2),
    'pct_abandono_test'    : round(float(y_test.mean()*100), 2),
    'ratio_desbalance'     : round(float(ratio_desbalance), 2),
    'n_features_entrada'   : int(n_cols - 1),
    'n_features_final'     : int(len(features)),
    'features_finales'     : features,
    'vars_log1p'           : VARS_LOG1P,
    'vars_informativas'    : VARS_INFORMATIVAS,
    'decisiones_escalado'  : decisiones,
    'cols_robust'          : cols_robust,
    'cols_standard'        : cols_standard,
    'cols_minmax'          : cols_minmax,
    'cols_ninguno'         : cols_ninguno,
    'validacion_ks'        : resultados_ks,
    'pipeline_pkl'         : str(ruta_pipeline.name),
    'archivos_generados'   : [
        'X_train.parquet', 'X_test.parquet',
        'y_train.parquet', 'y_test.parquet',
        'X_train_prep.parquet', 'X_test_prep.parquet',
        'pipeline_preprocesamiento.pkl', 'meta_preparacion.json'
    ]
}

# Convertir tipos numpy a tipos Python nativos para JSON
def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    elif hasattr(obj, 'item'):   # numpy scalar (int64, float64, bool_...)
        return obj.item()
    elif isinstance(obj, bool):
        return bool(obj)
    return obj

ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
with open(ruta_meta, 'w', encoding='utf-8') as f:
    json.dump(to_json_safe(meta), f, indent=2, ensure_ascii=False)

print('=' * 65)
print('ARCHIVOS GUARDADOS EN data/05_modelado/')
print('=' * 65)
for archivo in meta['archivos_generados']:
    ruta_arch = RUTA_MODELADO / archivo
    tam = ruta_arch.stat().st_size / 1024 if ruta_arch.exists() else 0
    print(f'   ✅ {archivo:<45} {tam:>7.1f} KB')

ARCHIVOS GUARDADOS EN data/05_modelado/
   ✅ X_train.parquet                                 533.0 KB
   ✅ X_test.parquet                                  151.1 KB
   ✅ y_train.parquet                                 164.0 KB
   ✅ y_test.parquet                                   41.0 KB
   ✅ X_train_prep.parquet                            540.6 KB
   ✅ X_test_prep.parquet                             158.1 KB
   ✅ pipeline_preprocesamiento.pkl                     5.2 KB
   ✅ meta_preparacion.json                             8.5 KB


In [12]:
# ============================================================================
# CELDA 12: GENERAR HTML
# ============================================================================
# Informe completo con tabla de decisiones de preprocesamiento,
# justificación de imputación informativa, validación KS y multicolinealidad.
# Toda la información técnica visible para el tribunal.
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_FASE5 / 'm01a_preparacion.html'

# ── KPIs ──────────────────────────────────────────────────────────────────────
kpis = [
    {'valor': fmt(len(X)),         'titulo': 'Registros totales'},
    {'valor': fmt(len(X_train)),   'titulo': 'Train (80%)'},
    {'valor': fmt(len(X_test)),    'titulo': 'Test (20%)'},
    {'valor': str(len(features)),  'titulo': 'Features finales'},
    {'valor': f'{y_train.mean()*100:.1f}%', 'titulo': 'Abandono train'},
    {'valor': f'{ratio_desbalance:.2f}:1',  'titulo': 'Ratio desbalance'},
]
kpis_html = ''.join(
    f'<div class="kpi"><span class="kpi-valor">{k["valor"]}</span>'
    f'<span class="kpi-titulo">{k["titulo"]}</span></div>'
    for k in kpis
)

# ── Tabla de decisiones de escalado ──────────────────────────────────────────
colores_esc = {'robust':'#fff5f5','standard':'#ebf8ff','minmax':'#f0fff4','ninguno':'#f7fafc'}
filas_dec = ''
for col, d in decisiones.items():
    bg = colores_esc.get(d['escalador'], '#fff')
    filas_dec += (
        f'<tr style="background:{bg}">'
        f'<td><code>{col}</code></td>'
        f'<td><strong>{d["escalador"].upper()}</strong></td>'
        f'<td>{d["skew"]}</td><td>{d["rango"]}</td><td>{d["nuniq"]}</td>'
        f'<td style="font-size:12px">{d["razon"]}</td></tr>'
    )

# ── Tabla KS ──────────────────────────────────────────────────────────────────
filas_ks = ''
for r in resultados_ks:
    bg = '#f0fff4' if r['ok'] else '#fff5f5'
    icono = '✅' if r['ok'] else '⚠️'
    filas_ks += (f'<tr style="background:{bg}">'
                 f'<td><code>{r["feature"]}</code></td>'
                 f'<td>{r["ks"]}</td><td>{r["p"]}</td><td>{icono}</td></tr>')

# ── Tabla multicolinealidad ───────────────────────────────────────────────────
filas_mc = ''
for par in PARES_CORRELADOS:
    bg = '#fff5f5' if par['r'] >= 0.7 else '#fffbeb'
    filas_mc += (
        f'<tr style="background:{bg}">'
        f'<td>{par["nivel"]}</td>'
        f'<td><code>{par["a"]}</code></td><td><code>{par["b"]}</code></td>'
        f'<td><strong>{par["r"]}</strong></td>'
        f'<td style="font-size:12px">{par["decision"]}</td></tr>'
    )

contenido = f'''
<section class="seccion">
  <h2>Dataset de entrada</h2>
  <p>Dataset <strong>D_strict</strong> — producido por Fase 3 tras auditoría de data leakage.
  {fmt(len(X))} expedientes × {len(features)} features + target <code>abandono</code>.
  Sin variables contaminadas, sin leakage temporal.</p>
  <div class="kpis-grid">{kpis_html}</div>
</section>

<section class="seccion">
  <h2>Distribución del target</h2>
  <p>Validación de que el split estratificado mantiene proporciones consistentes.
  Ratio de desbalance {ratio_desbalance:.2f}:1 — moderado, gestionado con
  <code>class_weight='balanced'</code> sin necesidad de SMOTE agresivo.</p>
  <img src="data:image/png;base64,{graficos_b64['distribucion_target']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Imputación informativa — nulos no aleatorios (MNAR)</h2>
  <p><strong>Concepto clave:</strong> Para <code>nota_1er_anio</code>, <code>nota_acceso</code>
  y <code>nota_selectividad</code> el valor nulo no es aleatorio — indica que el alumno
  accedió por una vía alternativa (FP, mayores de 25, traslado). El nulo <em>en sí mismo</em>
  es información predictiva sobre el perfil del estudiante.</p>
  <p>Estrategia: imputación con mediana de quienes sí tienen nota +
  indicador binario <code>_missing</code> que el modelo puede aprender a usar.
  Esto se denomina <strong>imputación informativa</strong> y preserva la señal
  que una imputación ciega destruiría.</p>
  <table class="tabla-datos">
    <thead><tr><th>Variable</th><th>Nulos</th><th>%</th><th>Estrategia</th></tr></thead>
    <tbody>
    {''.join(f"<tr><td><code>{v}</code></td><td>{fmt(df[v].isnull().sum())}</td><td>{df[v].isnull().mean()*100:.1f}%</td><td>Mediana + indicador <code>{v}_missing</code></td></tr>" for v in VARS_INFORMATIVAS if df[v].isnull().sum()>0)}
    <tr><td><code>max_pagos</code></td><td>3</td><td>0.0%</td><td>Mediana simple (MCAR)</td></tr>
    </tbody>
  </table>
</section>

<section class="seccion">
  <h2>Transformación log1p — variables con skew extremo</h2>
  <p>Variables con |skew| &gt; 3 tienen distribuciones tan asimétricas que ni RobustScaler
  las normaliza completamente. La transformación <code>log1p(x) = log(1+x)</code> reduce
  la asimetría antes del escalado. Se usa <code>log1p</code> (no <code>log</code>) porque
  hay ceros — <code>log(0) = -∞</code>.</p>
  <table class="tabla-datos">
    <thead><tr><th>Variable</th><th>Justificación</th></tr></thead>
    <tbody>
    {''.join(f"<tr><td><code>{v}</code></td><td>{{'orden_preferencia':'skew=3.64 — mayoría elige 1ª opción, cola larga','cred_repetidos':'skew=3.04 — mayoría no repite, outliers extremos','tasa_repeticion':'skew=3.04 — patrón idéntico a créditos repetidos','edad_entrada':'skew=3.29 — alumnos mayores son outliers reales del dominio','anios_gap':'skew=9.78 — casi todos 0, algunos con años de pausa','n_anios_sin_notas':'skew=3.05 — mayoría con presencia académica continua'}}.get(v, '')}}</td></tr>" for v in VARS_LOG1P)}
    </tbody>
  </table>
</section>

<section class="seccion">
  <h2>Tabla de decisiones de preprocesamiento</h2>
  <p>Cada feature recibe el escalador óptimo según sus estadísticos reales.
  Criterios: <strong>RobustScaler</strong> si |skew|&gt;2 o rango&gt;50 —
  <strong>StandardScaler</strong> si distribución aceptable —
  <strong>MinMaxScaler</strong> si variable acotada en [0,1] —
  <strong>Sin escalado</strong> si binaria.</p>
  <table class="tabla-datos">
    <thead>
      <tr><th>Feature</th><th>Escalador</th><th>|Skew|</th><th>Rango</th><th>Únicos</th><th>Justificación</th></tr>
    </thead>
    <tbody>{filas_dec}</tbody>
  </table>
</section>

<section class="seccion">
  <h2>Efecto del pipeline adaptativo</h2>
  <img src="data:image/png;base64,{graficos_b64['efecto_escalado']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Validación estadística del split — Test Kolmogorov-Smirnov</h2>
  <p>Test KS para las features más importantes: H₀ = train y test tienen la misma distribución.
  <strong>p &gt; 0.05 confirma que el split es estadísticamente representativo</strong> y que
  los resultados en test son generalizables a nueva población.</p>
  <table class="tabla-datos">
    <thead><tr><th>Feature</th><th>KS statistic</th><th>p-valor</th><th>Resultado</th></tr></thead>
    <tbody>{filas_ks}</tbody>
  </table>
</section>

<section class="seccion">
  <h2>Multicolinealidad — pares correlados y decisiones</h2>
  <p>Pares con |r| ≥ 0.5 detectados en Fase 4. Se documentan las decisiones explícitas
  para cada par. Árboles y boosting son inmunes por diseño. Para modelos lineales
  se aplica regularización L2 que gestiona la colinealidad sin eliminar variables.</p>
  <table class="tabla-datos">
    <thead><tr><th>Nivel</th><th>Variable A</th><th>Variable B</th><th>r</th><th>Decisión</th></tr></thead>
    <tbody>{filas_mc}</tbody>
  </table>
</section>
'''

render_pagina_desde_fichero('f5_m01a_preparacion.ipynb', contenido, RUTA_HTML_SALIDA)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m01a_preparacion.html


In [13]:
# ============================================================================
# CELDA 13: RESUMEN FINAL
# ============================================================================

print('=' * 65)
print('RESUMEN FINAL — f5_m01a_preparacion')
print('=' * 65)
print(f'Dataset    : dataset_final_tfm.parquet (D_strict) — data/03_features/')
print(f'Registros  : {fmt(len(X))} → train {fmt(len(X_train))} / test {fmt(len(X_test))}')
print(f'Features   : {len(features)} (incl. {len(VARS_INFORMATIVAS)} indicadores _missing)')
print(f'Log1p      : {len(VARS_LOG1P)} variables transformadas')
print(f'Escalado   : {len(cols_robust)} Robust + {len(cols_standard)} Standard + {len(cols_minmax)} MinMax + {len(cols_ninguno)} sin escalar')
print(f'Target     : abandono — train={y_train.mean()*100:.1f}% / test={y_test.mean()*100:.1f}%')
print(f'Desbalance : {ratio_desbalance:.2f}:1 — moderado → class_weight=balanced')
print(f'Split KS   : {sum(r["ok"] for r in resultados_ks)}/{len(resultados_ks)} features OK')
print()
print('📁 Archivos en data/05_modelado/:')
print('   X_train/test.parquet          → splits sin preprocesar')
print('   X_train_prep/test_prep.parquet → splits preprocesados')
print('   pipeline_preprocesamiento.pkl  → pipeline serializado')
print('   meta_preparacion.json          → metadatos completos')
print()
print('✅ Dataset en data/03_features/ — ruta correcta')
print()
print('➡️  Siguiente: f5_m01b_lineales_basico.ipynb')

RESUMEN FINAL — f5_m01a_preparacion
Dataset    : dataset_final_tfm.parquet (D_strict) — data/03_features/
Registros  : 33.621 → train 26.896 / test 6.725
Features   : 27 (incl. 3 indicadores _missing)
Log1p      : 6 variables transformadas
Escalado   : 7 Robust + 14 Standard + 1 MinMax + 5 sin escalar
Target     : abandono — train=29.2% / test=29.2%
Desbalance : 2.42:1 — moderado → class_weight=balanced
Split KS   : 7/8 features OK

📁 Archivos en data/05_modelado/:
   X_train/test.parquet          → splits sin preprocesar
   X_train_prep/test_prep.parquet → splits preprocesados
   pipeline_preprocesamiento.pkl  → pipeline serializado
   meta_preparacion.json          → metadatos completos

✅ Dataset en data/03_features/ — ruta correcta

➡️  Siguiente: f5_m01b_lineales_basico.ipynb
